# Hydrological Drought Event Detection

This notebook identifies and characterises **hydrological drought events** from the daily SSI series computed in `SSI_transformation.ipynb`.

## Pipeline

For each of the 33 gauging stations:

1. **Flag drought days** — SSI < −1.28 (P10 of the standard normal distribution).
2. **Identify raw events** — group consecutive drought days into candidate events.
3. **Pool events** — merge two events if the gap between them is ≤ 10 days (short recovery periods that do not represent a true end of drought are absorbed into the surrounding event).
4. **Filter events** — discard events whose total span is < 5 days (noise removal).
5. **Compute event characteristics** — for each surviving event extract the metrics described below.

> **Ordering note:** pooling is applied *before* the minimum-duration filter. This is the methodologically correct order (Yevjevich 1967, run theory): filtering first would remove short sub-events before they have the chance to be merged into a longer valid event.

## Output event characteristics

| Column | Definition |
|---|---|
| `station_id` | Gauging station identifier |
| `event_id` | Sequential event index per station |
| `start_date` | First day below threshold |
| `end_date` | Last day below threshold (span includes gap days in pooled events) |
| `duration` | Total days from `start_date` to `end_date` inclusive |
| `severity` | Σ \|threshold − SSI\| for drought days only (positive, deficit area) |
| `intensity` | severity / duration |
| `peak_SSI` | Minimum SSI reached during the event |
| `peak_date` | Date of minimum SSI |

In [ ]:
# ── Imports and global constants ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# P10 threshold: SSI < -1.28 defines a drought day
THRESHOLD    = -1.28

# Minimum event duration (total span). Events shorter than this are discarded.
MIN_DURATION = 5   # days

# Maximum inter-event gap for pooling. Events separated by <= this are merged.
POOL_GAP     = 10  # days

In [ ]:
# ── Load SSI data ──────────────────────────────────────────────────────────────
# Read the daily SSI file produced by SSI_transformation.ipynb.
# We parse dates and sort to guarantee chronological order per station.
df = pd.read_csv(
    'data/SSI_daily.csv',
    parse_dates=['date']
)
df = df.sort_values(['station_id', 'date']).reset_index(drop=True)

print(f"Rows       : {len(df):,}")
print(f"Stations   : {df['station_id'].nunique()}")
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"NaN SSI    : {df['SSI'].isna().sum():,} ({df['SSI'].isna().mean()*100:.2f}%)")
df.head()

## Helper functions

Three focused functions implement the detection pipeline:

- `detect_raw_events` — scans the SSI series and returns the integer positions of every consecutive run below the threshold.
- `pool_events` — merges adjacent events whose gap (in calendar days) is ≤ `POOL_GAP`.
- `filter_events` — removes events whose total span is < `MIN_DURATION`.
- `compute_event_stats` — calculates all output metrics for each surviving event.

**NaN handling:** days with NaN SSI are treated as non-drought days (conservative assumption). They may split what would otherwise be a single event; the pooling step will then re-merge them if the resulting gap is ≤ 10 days.

In [ ]:
# ── Step 1 – Flag and group consecutive drought days ──────────────────────────
def detect_raw_events(ssi_series, threshold=THRESHOLD):
    """
    Identify runs of consecutive days with SSI < threshold.

    Parameters
    ----------
    ssi_series : pd.Series (integer-indexed, chronologically ordered)
    threshold  : float

    Returns
    -------
    list of (start_pos, end_pos) integer index pairs (inclusive)
    """
    # NaN counts as False (not drought) via pandas comparison
    below = (ssi_series < threshold).values
    events = []
    in_event = False

    for i, flag in enumerate(below):
        if flag and not in_event:
            in_event = True
            start = i
        elif not flag and in_event:
            in_event = False
            events.append((start, i - 1))

    # Close an event that runs to the last record
    if in_event:
        events.append((start, len(below) - 1))

    return events

In [ ]:
# ── Step 2 – Pool events with short inter-event gaps ─────────────────────────
def pool_events(events, dates, gap_days=POOL_GAP):
    """
    Merge consecutive events separated by <= gap_days non-drought days.

    Parameters
    ----------
    events   : list of (start_pos, end_pos)
    dates    : pd.DatetimeIndex aligned with the SSI series
    gap_days : int — maximum inter-event gap to trigger merging

    Returns
    -------
    list of (start_pos, end_pos) after merging
    """
    if not events:
        return []

    pooled = [list(events[0])]  # mutable copy of first event

    for start, end in events[1:]:
        prev_end = pooled[-1][1]
        # Calendar days between end of previous event and start of current event
        # (exclusive of both endpoints = number of non-drought days in gap)
        gap = (dates[start] - dates[prev_end]).days - 1

        if gap <= gap_days:
            # Absorb current event into the previous one
            pooled[-1][1] = end
        else:
            pooled.append([start, end])

    return [tuple(e) for e in pooled]

In [ ]:
# ── Step 3 – Filter short events ─────────────────────────────────────────────
def filter_events(events, dates, min_days=MIN_DURATION):
    """
    Discard events whose total span (start to end inclusive) is < min_days.

    Parameters
    ----------
    events   : list of (start_pos, end_pos)
    dates    : pd.DatetimeIndex
    min_days : int

    Returns
    -------
    filtered list of (start_pos, end_pos)
    """
    return [
        (s, e) for s, e in events
        if (dates[e] - dates[s]).days + 1 >= min_days
    ]

In [ ]:
# ── Step 4 – Compute event characteristics ───────────────────────────────────
def compute_event_stats(events, ssi_series, dates, station_id, threshold=THRESHOLD):
    """
    Calculate output metrics for each event.

    Parameters
    ----------
    events     : list of (start_pos, end_pos)
    ssi_series : pd.Series (integer-indexed)
    dates      : pd.DatetimeIndex
    station_id : station identifier
    threshold  : float

    Returns
    -------
    list of dicts, one per event
    """
    records = []

    for event_id, (start, end) in enumerate(events, start=1):
        event_ssi   = ssi_series.iloc[start : end + 1]
        event_dates = dates[start : end + 1]

        # Total span: start_date to end_date inclusive (includes gap days if pooled)
        duration = (dates[end] - dates[start]).days + 1

        # Severity: sum of deficit on drought days only (positive values)
        drought_mask = event_ssi < threshold
        drought_ssi  = event_ssi[drought_mask]
        severity     = float(np.sum(np.abs(threshold - drought_ssi)))

        # Intensity: severity normalised by total event duration
        intensity = severity / duration

        # Peak: worst SSI day
        peak_pos  = event_ssi.idxmin()   # integer label in original series
        peak_ssi  = float(event_ssi.min())
        peak_date = dates[peak_pos]

        records.append({
            'station_id': station_id,
            'event_id'  : event_id,
            'start_date': dates[start],
            'end_date'  : dates[end],
            'duration'  : duration,
            'severity'  : round(severity,  4),
            'intensity' : round(intensity, 4),
            'peak_SSI'  : round(peak_ssi,  4),
            'peak_date' : peak_date,
        })

    return records

## Main loop

Apply the full pipeline to every station. The four steps are executed sequentially per station and the results are collected into a single list.

In [ ]:
# ── Main detection loop over all 33 stations ──────────────────────────────────
all_records = []

for station in tqdm(df['station_id'].unique(), desc='Detecting drought events'):

    # Extract station slice, reset index so positions are 0-based
    sub   = df[df['station_id'] == station].sort_values('date').reset_index(drop=True)
    ssi   = sub['SSI']
    dates = pd.DatetimeIndex(sub['date'])

    # 1. Raw events: consecutive drought days
    raw = detect_raw_events(ssi)

    # 2. Pool: merge events with inter-event gap <= POOL_GAP days
    pooled = pool_events(raw, dates)

    # 3. Filter: remove events with total span < MIN_DURATION days
    filtered = filter_events(pooled, dates)

    # 4. Compute characteristics
    records = compute_event_stats(filtered, ssi, dates, station)
    all_records.extend(records)

print(f"\nTotal drought events detected: {len(all_records):,}")

In [ ]:
# ── Assemble final DataFrame and quick sanity check ───────────────────────────
events_df = pd.DataFrame(all_records)

# Enforce column order
events_df = events_df[[
    'station_id', 'event_id',
    'start_date', 'end_date', 'duration',
    'severity', 'intensity',
    'peak_SSI', 'peak_date'
]]

print(events_df.dtypes)
print()
print(events_df.describe())
print()
print(f"Events per station (mean): {events_df.groupby('station_id').size().mean():.1f}")
print(f"Longest event : {events_df['duration'].max()} days")
print(f"Most severe   : {events_df['severity'].max():.2f}")
print(f"Lowest peak   : {events_df['peak_SSI'].min():.4f}")
events_df.head(10)

## Volume severity (hm³)

The SSI-based `severity` column measures the standardised deficit area (dimensionless).  
Here we add a physically meaningful companion: the **volumetric deficit below the P10 climatological discharge threshold**.

### Method

1. **P10 threshold** — for every station × DOY, gather all observed Q values within a ±15-day window across all years (same pooling as the SSI distribution fitting) and compute the 10th percentile. This produces a seasonally-varying threshold Q̂ₚ₁₀(station, DOY).

2. **Event window** — start/end dates are taken directly from the SSI-detected events (`events_df`). No re-detection is performed from Q.

3. **Daily deficit** — for each calendar day inside an event span:  
   deficit(day) = max(0, Q̂ₚ₁₀(station, DOY(day)) − Q_obs(day))

4. **Volume conversion** — summing daily deficits:  
   1 m³/s · day = 86 400 m³ = **0.0864 hm³**

In [ ]:
# ── Compute P10 climatological Q threshold (per station × DOY) ────────────────
# Uses the same ±15-day pooling window as the SSI distribution fitting.

Q_raw = pd.read_csv(
    'data/caudales_diarios_imputados_CORE_FILTRADO.csv',
    parse_dates=['date']
)
Q_raw = (Q_raw[['station_id', 'date', 'Q_imp']]
         .sort_values(['station_id', 'date'])
         .reset_index(drop=True))
Q_raw['doy'] = Q_raw['date'].dt.dayofyear

WINDOW = 15  # ±15-day pooling window (mirrors SSI fitting)

stations  = Q_raw['station_id'].unique()
doy_range = np.arange(1, 367)   # DOY 1–366 (covers leap years)

q_p10_records = []

for station in tqdm(stations, desc='Computing Q P10 thresholds'):
    sub    = Q_raw[Q_raw['station_id'] == station]
    Q_vals = sub['Q_imp'].values
    doys   = sub['doy'].values

    for doy in doy_range:
        lo = doy - WINDOW
        hi = doy + WINDOW

        # Wrap-around window at year boundaries
        if lo < 1:
            mask = (doys >= lo + 366) | (doys <= hi)
        elif hi > 366:
            mask = (doys >= lo) | (doys <= hi - 366)
        else:
            mask = (doys >= lo) & (doys <= hi)

        pool = Q_vals[mask]
        pool = pool[~np.isnan(pool)]
        p10  = float(np.percentile(pool, 10)) if len(pool) > 0 else np.nan

        q_p10_records.append({'station_id': station, 'doy': doy, 'Q_p10': p10})

q_p10 = pd.DataFrame(q_p10_records)
print(f"P10 table: {q_p10.shape}  (expected {len(stations)} stations × 366 DOYs = {len(stations)*366} rows)")
print(q_p10.head(10))

In [ ]:
# ── Compute volume severity (hm³) for each SSI-detected event ─────────────────
# Event windows (start_date / end_date) come from events_df — no re-detection.
# Daily deficit = max(0, Q_p10(station, DOY) - Q_obs(day))
# Volume: Σ deficit × 86400 s/day / 1e6 m³/hm³  =  Σ deficit × 0.0864  hm³

M3S_TO_HM3_DAY = 86_400 / 1e6   # 0.0864 hm³ per (m³/s · day)

# Merge P10 threshold into Q_raw once for efficient lookup
Q_lookup = Q_raw.merge(q_p10, on=['station_id', 'doy'], how='left')
# Pre-index by station for fast slicing
Q_lookup = Q_lookup.set_index('station_id')

severity_hm3_list = []

for _, row in tqdm(events_df.iterrows(), total=len(events_df), desc='Computing volume severity'):
    station    = row['station_id']
    start_date = row['start_date']
    end_date   = row['end_date']

    sub = Q_lookup.loc[station]
    # Select days within the SSI event window
    mask   = (sub['date'] >= start_date) & (sub['date'] <= end_date)
    window = sub[mask]

    # Daily deficit (only on days below the P10 threshold)
    deficit    = np.maximum(0.0, window['Q_p10'] - window['Q_imp'])
    volume_hm3 = float(deficit.sum() * M3S_TO_HM3_DAY)
    severity_hm3_list.append(round(volume_hm3, 4))

events_df['severity_hm3'] = severity_hm3_list

# Re-enforce column order including the new metric
events_df = events_df[[
    'station_id', 'event_id',
    'start_date', 'end_date', 'duration',
    'severity', 'intensity',
    'peak_SSI', 'peak_date',
    'severity_hm3'
]]

print(f"severity_hm3 added")
print(f"  Range : {events_df['severity_hm3'].min():.4f} – {events_df['severity_hm3'].max():.2f} hm³")
print(f"  Median: {events_df['severity_hm3'].median():.4f} hm³")
print(f"  NaNs  : {events_df['severity_hm3'].isna().sum()}")
events_df.head(10)

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
out_path = 'data/SSI_drought_events.csv'
events_df.to_csv(out_path, index=False)
print(f"Saved {len(events_df):,} events to '{out_path}'")

In [ ]:
events_df.describe()

In [ ]:
events_df.groupby("station_id").size().describe()

In [ ]:
events_df.loc[events_df["peak_SSI"].idxmin()]

In [ ]:
(events_df["peak_SSI"] < -4).sum()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(events_df["peak_SSI"], bins=50)
plt.show()

In [ ]:
plt.hist(df['SSI'].dropna(), bins=100)
plt.show()

In [ ]:
"""
Generate the drought event summary statistics table (PDF + PNG).
Loads events_df directly from the saved CSV so the whole notebook
does not need to be re-executed.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load data ─────────────────────────────────────────────────────────────────
events_df = pd.read_csv('data/SSI_drought_events.csv', parse_dates=['start_date', 'end_date', 'peak_date'])

# ── Compute statistics ────────────────────────────────────────────────────────
cols  = ["duration", "severity", "intensity", "peak_SSI", "severity_hm3"]
stats = events_df[cols].describe(percentiles=[0.25, 0.5, 0.75])

def iqr(col):
    return events_df[col].quantile(0.75) - events_df[col].quantile(0.25)

# Row definitions: (Descriptor, Symbol, Units, mean, median, IQR, min, max)
rows = [
    ("Duration",      r"$D$",       "days",
     stats.loc["mean","duration"],      stats.loc["50%","duration"],
     iqr("duration"),                   stats.loc["min","duration"],      stats.loc["max","duration"]),

    ("Severity",      r"$S$",       "\u2014",
     stats.loc["mean","severity"],      stats.loc["50%","severity"],
     iqr("severity"),                   stats.loc["min","severity"],      stats.loc["max","severity"]),

    ("Intensity",     r"$I$",       r"day$^{-1}$",
     stats.loc["mean","intensity"],     stats.loc["50%","intensity"],
     iqr("intensity"),                  stats.loc["min","intensity"],     stats.loc["max","intensity"]),

    ("Peak SSI",      r"SSI$^*$",   "\u2014",
     stats.loc["mean","peak_SSI"],      stats.loc["50%","peak_SSI"],
     iqr("peak_SSI"),                   stats.loc["min","peak_SSI"],      stats.loc["max","peak_SSI"]),

    ("Vol. severity", r"$V$",       r"hm$^3$",
     stats.loc["mean","severity_hm3"],  stats.loc["50%","severity_hm3"],
     iqr("severity_hm3"),               stats.loc["min","severity_hm3"],  stats.loc["max","severity_hm3"]),
]

# ── Format helpers ─────────────────────────────────────────────────────────────
def fmt(val, descriptor):
    if descriptor == "Duration":
        return f"{val:.1f}"
    return f"{val:.3f}"

# ── Build cell data ────────────────────────────────────────────────────────────
col_labels = ["Descriptor", "Symbol", "Units", "Mean", "Median", "IQR", "Min", "Max"]
table_data = []
for desc, sym, unit, mean, med, iq, mn, mx in rows:
    table_data.append([
        desc, sym, unit,
        fmt(mean, desc), fmt(med, desc), fmt(iq, desc),
        fmt(mn, desc),   fmt(mx, desc)
    ])

# ── Style constants ────────────────────────────────────────────────────────────
HEADER_COLOR = "#2d3e50"
HEADER_TEXT  = "white"
ROW_COLORS   = ["#f5f7fa", "white"]
EDGE_COLOR   = "#c0c8d0"
FONT_BODY    = 10
FONT_HEADER  = 10.5

# Column widths (proportional)
col_widths = [0.16, 0.10, 0.10, 0.11, 0.11, 0.11, 0.11, 0.11]

n_rows     = len(table_data)
fig_width  = 11
row_height = 0.48
fig_height = (n_rows + 1.6) * row_height   # header + data rows + footnote

fig, ax = plt.subplots(figsize=(fig_width, fig_height))
ax.axis("off")

total_w = sum(col_widths)
col_x   = np.cumsum([0] + col_widths[:-1]) / total_w   # normalised left edges
col_w   = np.array(col_widths) / total_w

y_header = 1.0
row_h    = 1.0 / (n_rows + 0.6)
y_data   = [1.0 - (i + 1) * row_h for i in range(n_rows)]

# ── Header row ─────────────────────────────────────────────────────────────────
for j, (label, x, w) in enumerate(zip(col_labels, col_x, col_w)):
    rect = mpatches.FancyBboxPatch(
        (x, y_header - row_h), w, row_h,
        boxstyle="square,pad=0", linewidth=0,
        facecolor=HEADER_COLOR, transform=ax.transAxes, clip_on=False
    )
    ax.add_patch(rect)
    ha   = "left"    if j == 0 else "center"
    xpos = x + 0.012 if j == 0 else x + w / 2
    ax.text(xpos, y_header - row_h / 2, label,
            transform=ax.transAxes, ha=ha, va="center",
            fontsize=FONT_HEADER, fontweight="bold", color=HEADER_TEXT)

# ── Data rows ──────────────────────────────────────────────────────────────────
for i, (row_vals, y) in enumerate(zip(table_data, y_data)):
    bg = ROW_COLORS[i % 2]
    for j, (val, x, w) in enumerate(zip(row_vals, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, y - row_h), w, row_h,
            boxstyle="square,pad=0", linewidth=0.4,
            edgecolor=EDGE_COLOR, facecolor=bg,
            transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)
        ha   = "left"    if j == 0 else "center"
        fw   = "bold"    if j == 0 else "normal"
        xpos = x + 0.012 if j == 0 else x + w / 2
        ax.text(xpos, y - row_h / 2, val,
                transform=ax.transAxes, ha=ha, va="center",
                fontsize=FONT_BODY, fontweight=fw)

# ── Top & bottom border lines (drawn in axes-fraction coordinates) ─────────────
for y_line in [y_header, y_data[-1] - row_h]:
    ax.plot([0, 1], [y_line, y_line],
            color="#2d3e50", linewidth=1.5,
            transform=ax.transAxes, clip_on=False)

# ── Footnote ───────────────────────────────────────────────────────────────────
footnote = (
    r"$n = 2{,}625$ events across 33 gauging stations (1961-2020).  "
    r"IQR\,=\,interquartile range ($\mathrm{Q}_{75} - \mathrm{Q}_{25}$)."
)
ax.text(0, y_data[-1] - row_h - 0.04, footnote,
        transform=ax.transAxes, ha="left", va="top",
        fontsize=8.5, color="#444444", style="italic")

plt.tight_layout()
plt.savefig("output/figures/drought_event_summary_stats.pdf", dpi=300, bbox_inches="tight")
plt.savefig("output/figures/drought_event_summary_stats.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: drought_event_summary_stats.pdf  /  drought_event_summary_stats.png")
